In [2]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 9.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
import datasets
import huggingface_hub

print(datasets.__version__)
print(huggingface_hub.__version__)

5.0.1
1.23.0


In [4]:
pip install transformers shap datasets

In [5]:
#Import libraries
import datasets
import numpy as np
import transformers
import shap

In [25]:
#Load dataset

dataset=datasets.load_dataset("stanfordnlp/imdb", split='test')

#Shorten the strings to fit into the pipeline model
#Take the first 20 reviews, and for each review, keep only its first 500 characters.
short_data=[v[:500] for v in dataset['text'][:20]]

In [7]:
dataset[0]

{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as 

In [8]:
print(short_data[0])

I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn't match the background, and painfully one-dimensional characters cannot be overcome with a 'sci-fi' setting. (I'm sure there are those of you out there who think Babylon 5 is good sci-fi 


In [24]:
#display the dataset

from datasets import load_dataset
import pandas as pd

dataset=datasets.load_dataset("stanfordnlp/imdb", split='test')
#We are not selecting labels because we are using pre trained models
short_data=[v[:500] for v in dataset['text'][:20]]
df=pd.DataFrame(short_data, columns=['text'])
df.head()
# print(short_data)

,text
0,I love sci-fi and am willing to put up with a ...
1,"Worth the entertainment value of a rental, esp..."
2,its a totally average film with a few semi-alr...
3,STAR RATING: ***** Saturday Night **** Friday ...
4,"First off let me say, If you haven't enjoyed a..."


In [10]:
#Defining the NLP pipeline

classifier=transformers.pipeline("sentiment-analysis", return_all_scores=True)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [11]:
# Example texts for sentiment analysis

texts=["I love this movie", "I hated the ending of this film."]

#Perform sentiment analysis
results=classifier(texts)

print(results)

[{'label': 'POSITIVE', 'score': 0.9998766183853149}, {'label': 'NEGATIVE', 'score': 0.9996867179870605}]


In [12]:
#Sentiment Analysis for the first few datapoints
classifier(short_data[:6])

[{'label': 'NEGATIVE', 'score': 0.999616265296936},
 {'label': 'NEGATIVE', 'score': 0.6170627474784851},
 {'label': 'NEGATIVE', 'score': 0.9997100234031677},
 {'label': 'NEGATIVE', 'score': 0.992783784866333},
 {'label': 'POSITIVE', 'score': 0.996307373046875},
 {'label': 'NEGATIVE', 'score': 0.9966711401939392}]

In [13]:
#Find the model accuracy

from sklearn.metrics import accuracy_score

classifier_results=classifier(short_data[:20])

predicted_labels = [
    1 if result['label'] == 'POSITIVE' else 0
    for result in classifier_results
]

accuracy=accuracy_score(dataset[:20]['label'], predicted_labels)
print(accuracy)


0.9


In [14]:
predicted_labels

[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]

In [26]:
#Define SHAP expalainer

explainer=shap.Explainer(classifier)

In [27]:
import numpy as np

#Convert the list to a Numpy array
short_data_array=np.array(short_data)

print(short_data_array[1])

Worth the entertainment value of a rental, especially if you like action movies. This one features the usual car chases, fights with the great Van Damme kick style, shooting battles with the 40 shell load shotgun, and even terrorist style bombs. All of this is entertaining and competently handled but there is nothing that really blows you away if you've seen your share before.<br /><br />The plot is made interesting by the inclusion of a rabbit, which is clever but hardly profound. Many of the c


In [28]:
#Explain the predictions on first two explainers
shap_values=explainer(short_data[:2])

In [30]:
print(shap_values.shape)

(2, None, 2)


In [29]:
#Shap force plot and text plot
shap.plots.text(shap_values[:,:,"POSITIVE"])